In [9]:
import base64
import json
import math
import mimetypes
import os
import re
import subprocess

import yt_dlp


def _parse_time_to_seconds(value):
    """Convert `MM:SS`, `HH:MM:SS`, or seconds string to integer seconds."""
    value = (value or "").strip()
    if not value:
        return None

    if value.isdigit():
        return int(value)

    parts = value.split(":")
    if not all(part.isdigit() for part in parts):
        raise ValueError("時間格式錯誤，請使用 秒、MM:SS 或 HH:MM:SS")

    if len(parts) == 2:
        m, s = map(int, parts)
        return m * 60 + s
    if len(parts) == 3:
        h, m, s = map(int, parts)
        return h * 3600 + m * 60 + s

    raise ValueError("時間格式錯誤，請使用 秒、MM:SS 或 HH:MM:SS")


def _clip_audio_file(audio_path, start_sec, end_sec):
    """Clip an existing audio file with ffmpeg and replace the original file."""
    temp_path = f"{audio_path}.clip_tmp.opus"

    # Keep only audio stream when cutting. Metadata and cover are re-embedded later.
    cmd = [
        "ffmpeg",
        "-y",
        "-i",
        audio_path,
        "-ss",
        str(start_sec),
        "-to",
        str(end_sec),
        "-map",
        "0:a:0",
        "-c:a",
        "copy",
        temp_path,
    ]

    completed = subprocess.run(cmd, capture_output=True, text=True)
    if completed.returncode != 0:
        raise RuntimeError(
            "ffmpeg 剪輯失敗:\n"
            f"stdout:\n{completed.stdout}\n"
            f"stderr:\n{completed.stderr}"
        )

    os.replace(temp_path, audio_path)


def _find_thumbnail_file(stem_path):
    """Find downloaded thumbnail by common extensions."""
    for ext in (".jpg", ".jpeg", ".png", ".webp"):
        candidate = f"{stem_path}{ext}"
        if os.path.exists(candidate):
            return candidate
    return None


def _extract_loudnorm_json(ffmpeg_output):
    """Extract the first loudnorm JSON block from ffmpeg output."""
    decoder = json.JSONDecoder()
    for match in re.finditer(r"\{", ffmpeg_output):
        try:
            stats, _ = decoder.raw_decode(ffmpeg_output[match.start() :])
        except json.JSONDecodeError:
            continue

        if isinstance(stats, dict) and "input_i" in stats and "input_tp" in stats:
            return stats

    return None


def _parse_loudnorm_number(value):
    """Convert loudnorm JSON value (string/number) to finite float."""
    text = str(value).strip().lower().replace("db", "").strip()
    if text in {"inf", "+inf", "-inf", "infinity", "+infinity", "-infinity", "nan"}:
        raise ValueError("invalid loudnorm numeric value")
    return float(text)


def _build_replaygain_tags(audio_path):
    """Analyze final audio loudness and build ReplayGain tags."""
    replaygain_tags = {
        "replaygain_reference_loudness": "89 dB",
        "replaygain_track_gain": "0.00 dB",
        "replaygain_track_peak": "1.000000",
    }

    cmd = [
        "ffmpeg",
        "-hide_banner",
        "-i",
        audio_path,
        "-af",
        "loudnorm=I=-18:TP=-1.5:LRA=11:print_format=json",
        "-f",
        "null",
        "-",
    ]

    completed = subprocess.run(cmd, capture_output=True, text=True)
    if completed.returncode != 0:
        print("⚠️ ReplayGain 分析失敗，將使用預設值。")
        return replaygain_tags

    analysis_output = f"{completed.stderr}\n{completed.stdout}"
    stats = _extract_loudnorm_json(analysis_output)
    if not stats:
        print("⚠️ 找不到 ReplayGain 分析結果，將使用預設值。")
        return replaygain_tags

    try:
        input_i = _parse_loudnorm_number(stats.get("input_i"))
        input_tp = _parse_loudnorm_number(stats.get("input_tp"))

        # ReplayGain reference loudness 89 dB is commonly mapped to -18 LUFS.
        track_gain = -18.0 - input_i
        track_peak = max(0.0, math.pow(10.0, input_tp / 20.0))

        replaygain_tags["replaygain_track_gain"] = f"{track_gain:.2f} dB"
        replaygain_tags["replaygain_album_gain"] = f"{track_gain:.2f} dB"
        replaygain_tags["replaygain_track_peak"] = f"{track_peak:.6f}"
        replaygain_tags["replaygain_album_peak"] = f"{track_peak:.6f}"
    except (TypeError, ValueError, KeyError):
        print("⚠️ ReplayGain 解析失敗，將使用預設值。")

    return replaygain_tags


def _embed_metadata_and_cover_opus(audio_path, info, thumbnail_path=None):
    """Embed metadata, ReplayGain, and cover art into opus after clipping."""
    try:
        from mutagen.flac import Picture
        from mutagen.oggopus import OggOpus
    except ImportError as e:
        raise RuntimeError(
            "需要安裝 mutagen 才能在剪輯後重新嵌入 metadata/封面: pip install mutagen"
        ) from e

    audio = OggOpus(audio_path)
    if audio.tags is None:
        audio.add_tags()

    # Force fixed tags requested for this workflow.
    tags = {
        "title": info.get("track") or info.get("title"),
        "artist": info.get("artist") or info.get("uploader") or info.get("channel"),
        "album": "YouTube",
        "albumartist": "",
        "album artist": "",
        "date": "2026",
        "compilation": "1",
        "comment": info.get("webpage_url") or info.get("original_url"),
    }
    tags.update(_build_replaygain_tags(audio_path))

    for key, value in tags.items():
        if value is None:
            continue
        audio.tags[key] = [str(value)]

    if thumbnail_path and os.path.exists(thumbnail_path):
        pic = Picture()
        pic.type = 3  # Front cover
        pic.desc = "Cover"
        pic.mime = mimetypes.guess_type(thumbnail_path)[0] or "image/jpeg"
        with open(thumbnail_path, "rb") as f:
            pic.data = f.read()

        # Ogg/Opus uses base64-encoded FLAC picture block.
        audio.tags["metadata_block_picture"] = [base64.b64encode(pic.write()).decode("ascii")]

    audio.save()


def download_opus_audio(url, start_time=None, end_time=None):
    """
    Download YouTube audio as .opus with best available quality.

    Final workflow:
    1) Download and convert to Opus
    2) Clip selected segment
    3) Re-embed metadata and cover art
    """
    start_sec = None
    end_sec = None
    if start_time and end_time:
        start_sec = _parse_time_to_seconds(start_time)
        end_sec = _parse_time_to_seconds(end_time)

        if start_sec >= end_sec:
            raise ValueError("結束時間必須大於開始時間")

        print(f"✂️ Clip range: {start_sec}s -> {end_sec}s")

    ydl_opts = {
        # Prioritize the highest-quality audio stream.
        "format": "bestaudio[acodec=opus]/bestaudio",

        # Save with video title as filename.
        "outtmpl": "%(title)s.%(ext)s",

        # Download thumbnail for final cover embedding.
        "writethumbnail": True,

        # Avoid accidental playlist downloads.
        "noplaylist": True,

        # Step 1: convert to Opus only (metadata/cover will be embedded after clipping).
        "postprocessors": [
            {
                "key": "FFmpegExtractAudio",
                "preferredcodec": "opus",
                "preferredquality": "0",  # Best quality
            }
        ],

        # Keep logs visible.
        "quiet": False,
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=True)
            original_path = ydl.prepare_filename(info)

        stem_path = os.path.splitext(original_path)[0]
        output_audio_path = f"{stem_path}.opus"
        thumbnail_path = _find_thumbnail_file(stem_path)

        # Step 2: explicit clipping after download/conversion.
        if start_sec is not None and end_sec is not None:
            _clip_audio_file(output_audio_path, start_sec, end_sec)
            print("✅ Clip completed successfully.")

        # Step 3: re-embed metadata and cover after clipping.
        _embed_metadata_and_cover_opus(output_audio_path, info, thumbnail_path)
        print("✅ Metadata and cover embedded successfully.")

        print("✅ Download completed successfully.")
    except yt_dlp.utils.DownloadError as e:
        print(f"❌ Download error: {e}")
        print("💡 If you see YouTube JS-runtime warnings, install Node.js and try again.")
    except Exception as e:
        print(f"❌ Unexpected error: {e}")


if __name__ == "__main__":
    # ffmpeg must be installed and in PATH for conversion and clipping.
    video_url = input("Enter YouTube URL: ").strip()

    clip_mode = input("只下載片段嗎？(y/N): ").strip().lower() == "y"
    if clip_mode:
        start_time = input("開始時間 (秒 / MM:SS / HH:MM:SS): ").strip()
        end_time = input("結束時間 (秒 / MM:SS / HH:MM:SS): ").strip()
        download_opus_audio(video_url, start_time, end_time)
    else:
        download_opus_audio(video_url)

[youtube:tab] Extracting URL: https://www.youtube.com/watch?v=ahg-na2UThU&list=RDahg-na2UThU&start_radio=1
[youtube:tab] Downloading just the video ahg-na2UThU because of --no-playlist
[youtube] Extracting URL: https://www.youtube.com/watch?v=ahg-na2UThU
[youtube] ahg-na2UThU: Downloading webpage


[youtube] ahg-na2UThU: Downloading android vr player API JSON
[info] ahg-na2UThU: Downloading 1 format(s): 251
[info] Downloading video thumbnail 41 ...
[info] Writing video thumbnail 41 to: [조각집🎨] '사랑이 잘' IU Live Clip (With 혁오X선셋 롤러코스터).webp
[download] Destination: [조각집🎨] '사랑이 잘' IU Live Clip (With 혁오X선셋 롤러코스터).webm
[download] 100% of    4.40MiB in 00:00:01 at 2.96MiB/s   
[ExtractAudio] Destination: [조각집🎨] '사랑이 잘' IU Live Clip (With 혁오X선셋 롤러코스터).opus
Deleting original file [조각집🎨] '사랑이 잘' IU Live Clip (With 혁오X선셋 롤러코스터).webm (pass -k to keep)
✅ Metadata and cover embedded successfully.
✅ Download completed successfully.
